Дані

In [1]:
import os, re, gc, glob, time
import numpy as np, pandas as pd
import scipy.sparse as sp
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from xgboost import XGBClassifier
from scipy import stats
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import QuantileTransformer

os.makedirs("results", exist_ok=True)
dev = torch.device("cuda")

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"не знайшов {name}: перевір Add Input"
    return hits[0]

tr = pd.read_csv(find("train_transaction.csv"))
idn = pd.read_csv(find("train_identity.csv"))
for c in tr.select_dtypes("float64").columns:
    tr[c] = tr[c].astype("float32")
df = tr.merge(idn, on="TransactionID", how="left")
df["has_identity"] = df["TransactionID"].isin(idn["TransactionID"])
del tr

N = len(df)
ts, tid = df.TransactionDT.values, df.TransactionID.values
y = df.isFraud.values
w = (ts // 86400) // 7
m_tr, m_va, m_te = (w <= 17), (w >= 18) & (w <= 21), (w >= 22)
assert (m_tr.sum(), m_va.sum(), m_te.sum()) == (431125, 77090, 82325)
assert y.sum() == 20663
split_of = np.where(m_tr, 0, np.where(m_va, 1, 2))
order = np.lexsort((tid, ts)); rank = np.empty(N, np.int64); rank[order] = np.arange(N)

X = df.drop(columns=["TransactionID", "isFraud", "TransactionDT"]).copy()
X["hour"] = (df.TransactionDT // 3600) % 24
X["weekday"] = (df.TransactionDT // 86400) % 7
X["has_identity"] = X["has_identity"].astype("int8")
for c in X.select_dtypes(include=["object", "category"]).columns:
    codes, _ = pd.factorize(X[c])
    X[c] = np.where(codes == -1, np.nan, codes).astype("float32")
df["D1n"] = (df["D1"] - df["TransactionDT"] / 86400).round()      # після X: у таблицю не потрапляє

FP = ["DeviceInfo", "id_30", "id_31", "id_33"]                    # конфігурація пристрою
fp_has = df[FP].notna().all(axis=1).values
fp_str = df[FP[0]].astype(str).str.cat([df[c].astype(str) for c in FP[1:]], sep="|").values
real_dev = np.where(fp_has, fp_str, None).astype(object)

major = int(xgb.__version__.split(".")[0])
DEV = dict(tree_method="hist", device="cuda") if major >= 2 else dict(tree_method="gpu_hist")
print("готово:", X.shape, "| рядків з відбитком:", int(fp_has.sum()))

готово: (590540, 434) | рядків з відбитком: 71120


Світи з кільцями, граф, ознаки

In [2]:
# ---- ознаки таблиці для нейромереж: один раз ----
Xn = X.values.astype("float32")
grp = lambda p: [i for i, c in enumerate(X.columns) if re.fullmatch(p, c)]
nanf = np.stack([np.isnan(Xn[:, g]).mean(1) for g in
                 (grp(r"V\d+"), grp(r"D\d+"), grp(r"id_\d+"), grp(r"M\d+"))], 1).astype("float32")
qt = QuantileTransformer(n_quantiles=200, output_distribution="normal",
                         subsample=200_000, random_state=0).fit(Xn[m_tr])
XT = np.hstack([np.nan_to_num(qt.transform(Xn), nan=0.0).clip(-5, 5).astype("float32"), nanf])
del Xn; gc.collect()

# ---- кільця: перепризначення справжніх чесних рядків ----
P_WEEKS = {"train": (0, 17), "val": (18, 21), "test": (22, 26)}
SPLIT_ID = {"train": 0, "val": 1, "test": 2}

def build_ring_rows(span_days, per_week, ring_seed):
    rng = np.random.default_rng(ring_seed)
    used = np.zeros(N, bool); span = int(span_days * 86400)
    rows_l, rid_l, per_l = [], [], []; rid = 0
    for name, (w0, w1) in P_WEEKS.items():
        sel = (w >= w0) & (w <= w1)
        p0, p1 = int(ts[sel].min()), int(ts[sel].max())
        pool = np.flatnonzero((y == 0) & fp_has & sel)
        pool = pool[np.argsort(ts[pool], kind="stable")]; tpool = ts[pool]
        for _ in range(per_week * (w1 - w0 + 1)):
            m = int(rng.integers(4, 9)); chosen = None
            for _try in range(500):
                t0 = rng.uniform(p0, p1 - span)
                lo = np.searchsorted(tpool, t0, side="left")
                hi = np.searchsorted(tpool, t0 + span, side="right")
                cand = pool[lo:hi]; cand = cand[~used[cand]]
                if len(cand) < m: continue
                vc = pd.Series(fp_str[cand]).value_counts(); vc = vc[vc >= m]
                if len(vc) == 0: continue
                g = vc.index[rng.choice(len(vc), p=vc.values / vc.values.sum())]
                chosen = rng.choice(cand[fp_str[cand] == g], m, replace=False); break
            assert chosen is not None, "не знайшов групу для кільця: зменш per_week"
            used[chosen] = True
            rows_l += list(chosen); rid_l += [rid] * m; per_l += [name] * m; rid += 1
    return np.array(rows_l), np.array(rid_l), np.array(per_l)

# ---- мітки сусідів і ребра за ключами світу ----
def label_feats2(cols, W_days, DLd):
    valid = KEYDF[cols].notna().all(axis=1).values; idx = np.flatnonzero(valid)
    nk = np.full(N, np.nan, "float32"); fk = np.full(N, np.nan, "float32")
    for pos in KEYDF.loc[valid, cols].groupby(cols, sort=False).indices.values():
        rows = idx[pos]; rows = rows[np.argsort(rank[rows], kind="stable")]
        tt, ff = ts[rows], y2[rows]; cs = np.concatenate([[0], np.cumsum(ff)])
        a = np.searchsorted(tt, tt, side="left")
        hi = np.minimum(np.searchsorted(tt, tt - DLd * 86400, side="right"), a)
        lo = np.minimum(np.searchsorted(tt, tt - W_days * 86400, side="left"), hi)
        nk[rows] = hi - lo; fk[rows] = cs[hi] - cs[lo]
    return nk, fk

def build_edges2(cols, K, W_days):
    valid = KEYDF[cols].notna().all(axis=1).values; idx = np.flatnonzero(valid)
    S, D = [], []
    for pos in KEYDF.loc[valid, cols].groupby(cols, sort=False).indices.values():
        if len(pos) < 2: continue
        rows = idx[pos]; rows = rows[np.argsort(rank[rows], kind="stable")]
        tt = ts[rows]; n = len(rows)
        lo = np.maximum(np.arange(n) - K, np.searchsorted(tt, tt - W_days * 86400, side="left"))
        cnt = np.arange(n) - lo; tot = int(cnt.sum())
        if tot == 0: continue
        dst_l = np.repeat(np.arange(n), cnt)
        start = np.repeat(np.cumsum(cnt) - cnt, cnt)
        src_l = np.arange(tot) - start + np.repeat(lo, cnt)
        S.append(rows[src_l].astype(np.int32)); D.append(rows[dst_l].astype(np.int32))
    return np.concatenate(S), np.concatenate(D)

REL2 = {"u2": (["gcard", "addr1"], 50, 30),
        "u3": (["gcard", "addr1", "P_emaildomain"], 50, 30),
        "u5": (["gcard", "addr1", "P_emaildomain", "D1n"], 50, 30),
        "dk": (["dev"], 50, 30)}
R_NAMES = list(REL2); KMAX = {k: v[1] for k, v in REL2.items()}

def hand_feats():
    Fd = {}
    for rel in ("u2", "u3", "u5"):
        nk, fk = label_feats2(REL2[rel][0], 30, 7)
        Fd[f"{rel}_nk"], Fd[f"{rel}_fk"] = nk, fk
        Fd[f"{rel}_rate"] = (fk / np.where(nk > 0, nk, np.nan)).astype("float32")
    D, DC = ["dev"], ["dev", "gcard"]
    n30, _ = label_feats2(D, 30, 0); n30s, _ = label_feats2(DC, 30, 0)
    n7, _ = label_feats2(D, 7, 0);   n7s, _ = label_feats2(DC, 7, 0)
    Fd["dk_other30"], Fd["dk_other7"] = n30 - n30s, n7 - n7s
    nk, fk = label_feats2(D, 30, 7); nks, fks = label_feats2(DC, 30, 7)
    Fd["dk_nk"], Fd["dk_fk"] = nk - nks, fk - fks
    Fd["dk_rate"] = (Fd["dk_fk"] / np.where(Fd["dk_nk"] > 0, Fd["dk_nk"], np.nan)).astype("float32")
    return pd.DataFrame(Fd)

def check_hand(HF, n=50, seed=0):
    rng = np.random.default_rng(seed); bad = 0
    dv, gc_ = KEYDF["dev"].values, KEYDF["gcard"].values
    for i in rng.choice(np.flatnonzero(~np.isnan(dv)), n, replace=False):
        oth = (dv == dv[i]) & (gc_ != gc_[i])
        e30 = oth & (ts < ts[i]) & (ts >= ts[i] - 30 * 86400)
        e7 = oth & (ts < ts[i]) & (ts >= ts[i] - 7 * 86400)
        kn = oth & (ts <= ts[i] - 7 * 86400) & (ts >= ts[i] - 30 * 86400)
        g = HF.iloc[i]
        if (int(e30.sum()) != int(g["dk_other30"]) or int(e7.sum()) != int(g["dk_other7"]) or
                int(kn.sum()) != int(g["dk_nk"]) or int(y2[kn].sum()) != int(g["dk_fk"])): bad += 1
    k5 = KEYDF[["gcard", "addr1", "P_emaildomain", "D1n"]].values
    for i in rng.choice(np.flatnonzero(~np.isnan(k5).any(axis=1)), n, replace=False):
        same = (k5 == k5[i]).all(axis=1)
        kn = same & (ts <= ts[i] - 7 * 86400) & (ts >= ts[i] - 30 * 86400)
        if int(kn.sum()) != int(HF["u5_nk"].values[i]) or int(y2[kn].sum()) != int(HF["u5_fk"].values[i]): bad += 1
    return bad

def make_inputs(HF):
    cols_ = []
    for r in ("u2", "u3", "u5"):
        g = lambda k, r=r: np.nan_to_num(HF[f"{r}_{k}"].values)
        cols_ += [np.log1p(g("nk")), np.log1p(g("fk")), g("rate"), (g("nk") > 0).astype("float32")]
    g = lambda k: np.nan_to_num(HF[k].values)
    cols_ += [np.log1p(g("dk_other30")), np.log1p(g("dk_other7")), np.log1p(g("dk_nk")),
              np.log1p(g("dk_fk")), g("dk_rate"), HF["dk_other30"].notna().values.astype("float32")]
    M = np.stack(cols_, 1).astype("float32")
    M = (M - M[m_tr].mean(0)) / (M[m_tr].std(0) + 1e-6)
    return torch.from_numpy(np.hstack([XT, M])).half().to(dev)

def leak_test():
    sp_t = torch.as_tensor(split_of, device=dev); t_t = torch.as_tensor(ts, device=dev)
    rg_ = np.random.default_rng(0)
    for sp_, cap in [(0, 0), (1, 1)]:
        ids = torch.as_tensor(rg_.choice(np.flatnonzero(split_of == sp_), 4000, replace=False), device=dev)
        N1, M1, N2_, M2 = hops(ids, 10, 5)
        assert (sp_t[N1[M1]] <= cap).all() and (sp_t[N2_[M2]] <= cap).all(), "сусіди з майбутньої частини"
        assert (t_t[N1] <= t_t[ids][:, None, None])[M1].all(), "сусід пізніший за вузол (крок 1)"
        assert (t_t[N2_] <= t_t[N1][:, :, :, None, None])[M2].all(), "сусід пізніший за вузол (крок 2)"

def setup_world(span_days, ring_seed, per_week=10):
    global y2, is_ring, KEYDF, NBR, DEG, XG2, HF, XA, XB
    NBR = DEG = XG2 = HF = XA = XB = None
    gc.collect(); torch.cuda.empty_cache()
    rows, rid_a, per_a = build_ring_rows(span_days, per_week, ring_seed)
    R = len(rows)
    assert len(np.unique(rows)) == R and (y[rows] == 0).all() and fp_has[rows].all()
    assert pd.Series(fp_str[rows]).groupby(rid_a).nunique().max() == 1, "у кільці різні конфігурації пристрою"
    spread = pd.Series(ts[rows]).groupby(rid_a).agg(lambda s: s.max() - s.min())
    assert spread.max() <= span_days * 86400, "кільце розтягнуте довше за вікно"
    assert (split_of[rows] == pd.Series(per_a).map(SPLIT_ID).values).all(), "кільце в чужому періоді"
    is_ring = np.zeros(N, bool); is_ring[rows] = True
    y2 = y.copy(); y2[rows] = 1
    dev_all = real_dev.copy(); dev_all[rows] = np.array(["ring%d" % k for k in rid_a], dtype=object)
    dk_code, _ = pd.factorize(pd.Series(dev_all))
    gcard = df["card1"].values.astype("float64").copy(); gcard[rows] = 10_000_000.0 + np.arange(R)
    KEYDF = pd.DataFrame({"gcard": gcard, "addr1": df["addr1"].values.astype("float64"),
                          "P_emaildomain": X["P_emaildomain"].values.astype("float64"),
                          "D1n": df["D1n"].values.astype("float64"),
                          "dev": np.where(dk_code >= 0, dk_code, np.nan).astype("float64")})
    NBR, DEG = [], []
    for rel, (cols, K, W_) in REL2.items():
        s, d = build_edges2(cols, K, W_)
        assert (rank[s] < rank[d]).all() and (s != d).all()
        assert (split_of[s[split_of[d] == 0]] == 0).all()
        o = np.argsort(d, kind="stable"); ds, ss = d[o], s[o]
        cnt = np.bincount(d, minlength=N); start = np.cumsum(cnt) - cnt
        pos = np.arange(len(ds)) - start[ds]; assert pos.max() < K
        tab = np.full((N, K), -1, dtype=np.int32); tab[ds, pos] = ss
        NBR.append(torch.from_numpy(tab).to(dev)); DEG.append(torch.from_numpy(cnt.astype(np.int64)).to(dev))
    HF = hand_feats()
    bad = check_hand(HF); print("  розбіжностей ручних ознак з перерахунком «в лоб»:", bad); assert bad == 0
    assert np.nan_to_num(HF.loc[is_ring, "u2_nk"].values).max() == 0, "у членів кілець є історія тієї ж картки"
    lab_cols = [c for c in HF.columns if c.startswith(("u2_", "u3_", "u5_"))]
    XA = pd.concat([X, HF[lab_cols]], axis=1); XB = pd.concat([X, HF], axis=1)
    XG2 = make_inputs(HF)
    leak_test(); print("  тест на витік пройдено", flush=True)
print("світ описано")

світ описано


Моделі

In [3]:
def sample(nodes, r, s, gen=None):
    deg = DEG[r][nodes]
    u = torch.rand(nodes.shape[0], s, device=dev, generator=gen)
    idx = (u * deg.unsqueeze(1)).long().clamp_(max=KMAX[R_NAMES[r]] - 1)
    nb = NBR[r][nodes.unsqueeze(1), idx].long()
    valid = (deg > 0).unsqueeze(1).expand(-1, s) & (nb >= 0)
    return nb.clamp(min=0), valid

def hops(tgt, s1, s2, gen=None):
    B, R = tgt.shape[0], len(R_NAMES)
    n1, v1 = zip(*[sample(tgt, r, s1, gen) for r in range(R)])
    N1, M1 = torch.stack(n1, 1), torch.stack(v1, 1)
    n2, v2 = zip(*[sample(N1.reshape(-1), r, s2, gen) for r in range(R)])
    N2_ = torch.stack(n2, 1).view(B, R, s1, R, s2)
    M2 = torch.stack(v2, 1).view(B, R, s1, R, s2) & M1[:, :, :, None, None]
    return N1, M1, N2_, M2

class RelSAGE2(nn.Module):
    def __init__(self, F_in, H=128, R=4, drop=0.2):
        super().__init__()
        self.R = R
        self.enc = nn.Sequential(nn.Linear(F_in, H), nn.ReLU(), nn.Dropout(drop))
        self.W1s, self.W2s = nn.Linear(H, H), nn.Linear(H, H)
        self.W1n = nn.ModuleList([nn.Linear(H, H, bias=False) for _ in range(R)])
        self.W2n = nn.ModuleList([nn.Linear(H, H, bias=False) for _ in range(R)])
        self.out = nn.Sequential(nn.Dropout(drop), nn.Linear(H, 1))

    @staticmethod
    def mmean(e, m, dim):
        m = m.unsqueeze(-1).to(e.dtype)
        return (e * m).sum(dim) / m.sum(dim).clamp(min=1)

    def forward(self, XG, tgt, s1, s2, gen=None):
        B, R = tgt.shape[0], self.R
        N1, M1, N2_, M2 = hops(tgt, s1, s2, gen)
        enc = lambda ids: self.enc(XG[ids].float())
        e_t = enc(tgt)
        e1 = enc(N1.reshape(-1)).view(B, R, s1, -1)
        e2 = enc(N2_.reshape(-1)).view(B, R, s1, R, s2, -1)
        a2 = self.mmean(e2, M2, 4)
        h1_u = F.relu(self.W1s(e1) + sum(self.W1n[j](a2[:, :, :, j]) for j in range(R)))
        a1 = self.mmean(e1, M1, 2)
        h1_t = F.relu(self.W1s(e_t) + sum(self.W1n[j](a1[:, j]) for j in range(R)))
        ah = self.mmean(h1_u, M1, 2)
        h2 = F.relu(self.W2s(h1_t) + sum(self.W2n[j](ah[:, j]) for j in range(R)))
        return self.out(h2).squeeze(-1)

@torch.no_grad()
def predict_avg(model, XG, idx_np, s1, s2, seed, bs=2048):
    model.eval(); ids = torch.as_tensor(idx_np, device=dev)
    gen = torch.Generator(device=dev); gen.manual_seed(1234 + 100 * seed)
    return torch.cat([torch.sigmoid(model(XG, ids[i:i + bs], s1, s2, gen)).float().cpu()
                      for i in range(0, len(ids), bs)]).numpy()

def train_nn2(make_model, XG, seed, epochs, patience, bs=1024, s1=10, s2=5, lr=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    tr_i, va_i, te_i = np.flatnonzero(m_tr), np.flatnonzero(m_va), np.flatnonzero(m_te)
    model = make_model(XG.shape[1]).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    posw = torch.tensor(float((y2[tr_i] == 0).sum() / (y2[tr_i] == 1).sum()), device=dev)
    tri = torch.as_tensor(tr_i, device=dev); yt = torch.as_tensor(y2, device=dev).float()
    best, bad, state = -1.0, 0, None
    for ep in range(1, epochs + 1):
        model.train(); perm = tri[torch.randperm(len(tri), device=dev)]
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            loss = F.binary_cross_entropy_with_logits(model(XG, b, s1, s2), yt[b], pos_weight=posw)
            opt.zero_grad(); loss.backward(); opt.step()
        v = average_precision_score(y2[va_i], predict_avg(model, XG, va_i, s1, s2, seed))
        if v > best:
            best, bad = v, 0; state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state)
    return predict_avg(model, XG, te_i, s1, s2, seed), best

def fit_xgb2(Xf, seed, quick=False):
    spw2 = (y2[m_tr] == 0).sum() / (y2[m_tr] == 1).sum()
    kw = dict(n_estimators=200, learning_rate=0.3, max_depth=5, early_stopping_rounds=20) if quick else \
         dict(n_estimators=1000, learning_rate=0.1, max_depth=6, early_stopping_rounds=50)
    m = XGBClassifier(subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw2,
                      eval_metric="aucpr", random_state=seed, **kw, **DEV)
    m.fit(Xf[m_tr], y2[m_tr], eval_set=[(Xf[m_va], y2[m_va])], verbose=False)
    return m.predict_proba(Xf[m_te])[:, 1], m.best_score
print("моделі описано")

моделі описано


Чесна метрика (зважування за частотою конфігурації)

In [4]:
vc_all = pd.Series(fp_str[fp_has]).value_counts()
cfg_logf = np.full(N, np.nan)
cfg_logf[fp_has] = np.log(pd.Series(fp_str[fp_has]).map(vc_all).values)
FM = "кільця vs чесні з відбитком"

def match_weights():
    """чесні рядки з відбитком перезважуються під розподіл частоти конфігурації кілець (тест)"""
    te = np.flatnonzero(m_te); rg = is_ring[te]
    lf = cfg_logf[te]; keep_h = (~rg) & (y2[te] == 0) & fp_has[te] & (lf >= lf[rg].min()) & (lf <= lf[rg].max())
    edges = np.quantile(lf[rg], np.linspace(0, 1, 9)); edges[0], edges[-1] = -np.inf, np.inf
    b = np.clip(np.searchsorted(edges, lf, side="right") - 1, 0, 7)
    n_r = np.bincount(b[rg], minlength=8).astype(float); n_h = np.bincount(b[keep_h], minlength=8).astype(float)
    wb = np.where(n_h > 0, (n_r / n_r.sum()) * keep_h.sum() / np.maximum(n_h, 1), 0.0)
    wt = np.zeros(len(te)); wt[rg] = 1.0; wt[keep_h] = wb[b[keep_h]]
    return te, rg | keep_h, wt

def eval_sets(p):
    te, keep, wt = match_weights(); yt, rg_ = y2[te], is_ring[te]
    return {"усі": average_precision_score(yt, p),
            FM: average_precision_score(yt[keep], p[keep], sample_weight=wt[keep]),
            "лише справжні": average_precision_score(yt[~rg_], p[~rg_])}

def table_ratio(seed):
    te, keep, wt = match_weights(); base = wt[is_ring[te]].sum() / wt[keep].sum()
    p, _ = fit_xgb2(X, seed, quick=True)
    return eval_sets(p)[FM] / base
print("метрику описано")

метрику описано


Агрегати сусідів по пристрою і запуск (≈ 1,5 год без GNN)

In [5]:
Xv = X.values.astype("float32")                      # NaN збережено
ncol = Xv.shape[1]
AGG_NAMES = [f"dkagg_{c}" for c in X.columns]
NB_, NC_, NG_ = "XGB-B (+ ознаки пристрою)", "XGB-C (+ агрегати ознак сусідів по пристрою)", "GNN-B"

cands = glob.glob("/kaggle/input/**/ieee_cis_stress_test_v2.csv", recursive=True) + \
        glob.glob("results/ieee_cis_stress_test_v2.csv")
REF = pd.read_csv(cands[0]) if cands else None
print("референс GNN-B:", cands[0] if cands else "НЕ ЗНАЙДЕНО: GNN-B буде перенавчено (+ ≈ 40 хв)",
      "| рядків:", 0 if REF is None else len(REF))

def dk_aggregates():
    nb = NBR[R_NAMES.index("dk")].cpu().numpy()      # [N, 50], -1 = порожньо
    msk = (nb >= 0).ravel()
    rows = np.repeat(np.arange(N), nb.shape[1])[msk]; cols = nb.ravel()[msk]
    A = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(N, N))
    deg = np.asarray(A.sum(axis=1)).ravel().astype("float32")
    G = np.empty((N, ncol), np.float32)
    for c0 in range(0, ncol, 100):
        blk = Xv[:, c0:c0 + 100]; ok = ~np.isnan(blk)
        S = A @ np.nan_to_num(blk, nan=0.0); C = A @ ok.astype(np.float32)
        with np.errstate(invalid="ignore", divide="ignore"):
            G[:, c0:c0 + 100] = np.where(C > 0, S / C, np.nan)
    return G, deg

def check_agg(G, n=300, seed=0):
    nb = NBR[R_NAMES.index("dk")].cpu().numpy()
    rng = np.random.default_rng(seed); cand = np.flatnonzero((nb >= 0).sum(1) > 0); bad = 0
    for i in rng.choice(cand, n, replace=False):
        nbi = nb[i][nb[i] >= 0]
        assert (rank[nbi] < rank[i]).all(), "сусід не з минулого"
        for j in rng.choice(ncol, 5, replace=False):
            v = Xv[nbi, j].astype("float64")
            ref_ = np.nan if not np.isfinite(v).any() else float(np.nanmean(v))
            g = float(G[i, j])
            if not ((np.isnan(ref_) and np.isnan(g)) or abs(ref_ - g) <= 1e-3 * (1 + abs(ref_))): bad += 1
    return bad

RESULT_D = "results/ieee_cis_mechanism.csv"
ROWS_D = pd.read_csv(RESULT_D).to_dict("records") if os.path.exists(RESULT_D) else []

def run_mech(span_days, setting, seed):
    if any(r["setting"] == setting and r["seed"] == seed for r in ROWS_D):
        print(f"{setting}, seed {seed}: вже є, пропускаю"); return
    t0 = time.time(); setup_world(span_days, seed)
    print(f"  санітарна перевірка таблиці (після зважування): {table_ratio(seed):.2f}× (очікуємо ≈ 1,0-1,4)", flush=True)
    G, deg = dk_aggregates(); bad = check_agg(G)
    print("  розбіжностей агрегатів з перерахунком «в лоб»:", bad); assert bad == 0
    XC = pd.concat([XB, pd.DataFrame(G, columns=AGG_NAMES, index=XB.index),
                    pd.Series(deg, name="dkagg_deg", index=XB.index)], axis=1)
    del G; gc.collect()
    out = []
    for name, Xf in [(NB_, XB), (NC_, XC)]:
        p, v = fit_xgb2(Xf, seed); out.append({"модель": name, "seed": seed, "setting": setting, "val": v, **eval_sets(p)})
    del XC; gc.collect()
    if REF is not None:
        r0 = REF[(REF.setting == setting) & (REF.seed == seed) & (REF["модель"] == NB_)]
        if len(r0):
            d = abs(float(r0[FM].iloc[0]) - out[0][FM])
            print(f"  узгодженість XGB-B зі старим запуском: |Δ| = {d:.3f}" + ("  УВАГА: світ міг відрізнятись" if d > 0.03 else ""))
    else:
        p, v = train_nn2(lambda F_in: RelSAGE2(F_in, R=len(R_NAMES)), XG2, seed, epochs=8, patience=2)
        out.append({"модель": NG_, "seed": seed, "setting": setting, "val": v, **eval_sets(p)})
    ROWS_D.extend(out); pd.DataFrame(ROWS_D).to_csv(RESULT_D, index=False)
    print(f"{setting}, seed {seed}: {time.time()-t0:.0f} с | кільця vs чесні: " +
          " | ".join(f"{o['модель'][:5]} {o[FM]:.3f}" for o in out), flush=True)

for name, span in [("швидкі кільця (2 дні)", 2.0), ("повільні кільця (21 день)", 21.0)]:
    for sd in range(5):
        run_mech(span, name, sd)
print("ГОТОВО")

референс GNN-B: НЕ ЗНАЙДЕНО: GNN-B буде перенавчено (+ ≈ 40 хв) | рядків: 0
  розбіжностей ручних ознак з перерахунком «в лоб»: 0
  тест на витік пройдено


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [16:12:20] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  санітарна перевірка таблиці (після зважування): 1.10× (очікуємо ≈ 1,0-1,4)
  розбіжностей агрегатів з перерахунком «в лоб»: 0
швидкі кільця (2 дні), seed 0: 480 с | кільця vs чесні: XGB-B 0.629 | XGB-C 0.565 | GNN-B 0.825
  розбіжностей ручних ознак з перерахунком «в лоб»: 0
  тест на витік пройдено
  санітарна перевірка таблиці (після зважування): 1.05× (очікуємо ≈ 1,0-1,4)
  розбіжностей агрегатів з перерахунком «в лоб»: 0
швидкі кільця (2 дні), seed 1: 531 с | кільця vs чесні: XGB-B 0.549 | XGB-C 0.554 | GNN-B 0.810
  розбіжностей ручних ознак з перерахунком «в лоб»: 0
  тест на витік пройдено
  санітарна перевірка таблиці (після зважування): 1.13× (очікуємо ≈ 1,0-1,4)
  розбіжностей агрегатів з перерахунком «в лоб»: 0
швидкі кільця (2 дні), seed 2: 526 с | кільця vs чесні: XGB-B 0.544 | XGB-C 0.563 | GNN-B 0.914
  розбіжностей ручних ознак з перерахунком «в лоб»: 0
  тест на витік пройдено
  санітарна перевірка таблиці (після зважування): 1.44× (очікуємо ≈ 1,0-1,4)
  розбіжностей

Підсумок

In [6]:
D = pd.DataFrame(ROWS_D)
if REF is not None:
    D = pd.concat([D, REF[REF["модель"] == NG_][["setting", "seed", "модель", "val", "усі", FM, "лише справжні"]]], ignore_index=True)
for setting, gdf in D.groupby("setting"):
    print("\n==", setting, "==")
    display(gdf.groupby("модель")[["усі", FM, "лише справжні"]].agg(["mean", "std"]).reindex([NB_, NC_, NG_]).round(3))
    pv = gdf.pivot(index="seed", columns="модель", values=FM).dropna()
    A = pv[NG_] - pv[NC_]; t, p = stats.ttest_1samp(A, 0.0)
    closed = (pv[NC_] - pv[NB_]).mean() / (pv[NG_] - pv[NB_]).mean()
    v = "GNN зберігає перевагу" if A.mean() >= 0.10 else ("перевагу переважно пояснюють атрибути сусідів" if A.mean() < 0.05 else "часткове пояснення")
    print(f"GNN-B − XGB-C: {A.mean():+.3f} ± {A.std()/np.sqrt(len(A)):.3f} (SE), p = {p:.3f} → {v}")
    print(f"частка розриву XGB-B→GNN-B, закрита XGB-C: {closed*100:.0f} % (seed: {len(pv)})")
    for col in ("лише справжні", "усі"):
        pr = gdf.pivot(index="seed", columns="модель", values=col).dropna()
        print(f"XGB-C − XGB-B на «{col}»: {(pr[NC_] - pr[NB_]).mean():+.4f}")


== повільні кільця (21 день) ==


усі         \
                                               mean    std   
модель                                                       
XGB-B (+ ознаки пристрою)                     0.630  0.008   
XGB-C (+ агрегати ознак сусідів по пристрою)  0.624  0.007   
GNN-B                                         0.620  0.005   

                                             кільця vs чесні з відбитком  \
                                                                    mean   
модель                                                                     
XGB-B (+ ознаки пристрою)                                          0.564   
XGB-C (+ агрегати ознак сусідів по пристрою)                       0.551   
GNN-B                                                              0.812   

                                                    лише справжні         
                                                std          mean    std  
модель                                                                    
XGB-B (+ ознаки пристрою)                     0.081         0.623  0.004  
XGB-C (+ агрегати ознак сусідів по пристрою)  0.056         0.619  0.003  
GNN-B                                         0.041         0.580  0.006

GNN-B − XGB-C: +0.261 ± 0.025 (SE), p = 0.000 → GNN зберігає перевагу
частка розриву XGB-B→GNN-B, закрита XGB-C: -5 % (seed: 5)
XGB-C − XGB-B на «лише справжні»: -0.0034
XGB-C − XGB-B на «усі»: -0.0061

== швидкі кільця (2 дні) ==


усі         \
                                               mean    std   
модель                                                       
XGB-B (+ ознаки пристрою)                     0.637  0.005   
XGB-C (+ агрегати ознак сусідів по пристрою)  0.633  0.005   
GNN-B                                         0.623  0.003   

                                             кільця vs чесні з відбитком  \
                                                                    mean   
модель                                                                     
XGB-B (+ ознаки пристрою)                                          0.617   
XGB-C (+ агрегати ознак сусідів по пристрою)                       0.592   
GNN-B                                                              0.872   

                                                    лише справжні         
                                                std          mean    std  
модель                                                                    
XGB-B (+ ознаки пристрою)                     0.067         0.620  0.004  
XGB-C (+ агрегати ознак сусідів по пристрою)  0.046         0.617  0.004  
GNN-B                                         0.051         0.582  0.004

GNN-B − XGB-C: +0.280 ± 0.019 (SE), p = 0.000 → GNN зберігає перевагу
частка розриву XGB-B→GNN-B, закрита XGB-C: -10 % (seed: 5)
XGB-C − XGB-B на «лише справжні»: -0.0032
XGB-C − XGB-B на «усі»: -0.0043
